In [ ]:
!pip install ultralytics -q

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from pathlib import Path

KAGGLE_INPUT   = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

# Cerca dataset_sar tra tutti i dataset caricati
dataset_sar_path = None
for d in KAGGLE_INPUT.rglob("dataset_sar"):
    if d.is_dir():
        dataset_sar_path = d
        break

# Fallback: cerca images/train direttamente
if dataset_sar_path is None:
    for d in KAGGLE_INPUT.rglob("images"):
        if (d / "train").exists() and (d / "val").exists():
            dataset_sar_path = d.parent
            break

if dataset_sar_path is None:
    print("ERRORE: dataset non trovato. Contenuto di /kaggle/input:")
    for p in KAGGLE_INPUT.iterdir():
        print(f"  {p}")
        for sub in p.iterdir():
            print(f"    {sub}")
else:
    train_imgs = list((dataset_sar_path / "images" / "train").glob("*.jpg"))
    val_imgs   = list((dataset_sar_path / "images" / "val").glob("*.jpg"))
    print(f"[OK] Dataset: {dataset_sar_path}")
    print(f"     Train:   {len(train_imgs)} immagini")
    print(f"     Val:     {len(val_imgs)} immagini")

In [ ]:
yaml_content = f"""path: {dataset_sar_path}
train: images/train
val:   images/val

nc: 1
names:
  0: person

kpt_shape: [17, 3]
flip_idx: [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]
"""

yaml_path = KAGGLE_WORKING / "data.yaml"
with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(f"[OK] data.yaml scritto in: {yaml_path}")
print(yaml_content)

In [ ]:
# SESSIONE 1: lascia Nano e Small, commenta Large
# SESSIONE 2: commenta Nano e Small, decommenta Large

EXPERIMENTS = [
    #{
    #   "id":      "fase1_nano",
    #    "weights": "yolo11n-pose.pt",
    #    "label":   "YOLO11n-Pose — Nano",
    #    "epochs":  30,
    #    "batch":   32,
    #    "imgsz":   640,
    #},
    #{
    #    "id":      "fase1_small",
    #    "weights": "yolo11s-pose.pt",
    #    "label":   "YOLO11s-Pose — Small",
    #    "epochs":  30,
    #    "batch":   24,
    #    "imgsz":   640,
    #},
     {
         "id":      "fase1_large",
         "weights": "yolo11l-pose.pt",
         "label":   "YOLO11l-Pose — Large",
         "epochs":  30,
         "batch":   16,
         "imgsz":   640,
     },
]

COMMON_PARAMS = {
    "data":        str(yaml_path),
    "workers":     4,
    "patience":    15,
    "save":        True,
    "save_period": 10,
    "plots":       True,
    "val":         True,
    "verbose":     True,
    "augment":     True,
    "hsv_h":       0.015,
    "hsv_s":       0.3,
    "hsv_v":       0.4,
    "degrees":     10.0,
    "translate":   0.1,
    "scale":       0.5,
    "flipud":      0.3,
    "fliplr":      0.5,
    "mosaic":      1.0,
    "mixup":       0.1,
    "project":     str(KAGGLE_WORKING / "runs" / "fase1"),
    "exist_ok":    False,
}

print(f"[OK] {len(EXPERIMENTS)} esperimenti configurati:")
for e in EXPERIMENTS:
    print(f"     {e['label']} — batch={e['batch']} epochs={e['epochs']}")

In [ ]:
import json
from datetime import datetime
from ultralytics import YOLO

device      = "cuda:0" if torch.cuda.is_available() else "cpu"
all_results = []

for exp in EXPERIMENTS:
    exp_id   = exp["id"]
    label    = exp["label"]
    ts       = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_name = f"{exp_id}_{ts}"

    print("\n" + "="*60)
    print(f"  {label}")
    print(f"  Run: {run_name}")
    print(f"  Epoche: {exp['epochs']}  Batch: {exp['batch']}")
    print("="*60)

    try:
        model   = YOLO(exp["weights"])
        results = model.train(
            **COMMON_PARAMS,
            epochs  = exp["epochs"],
            batch   = exp["batch"],
            imgsz   = exp["imgsz"],
            device  = device,
            name    = run_name,
        )

        m = {}
        if hasattr(results, "results_dict"):
            rd = results.results_dict
            m["box_precision"]  = float(rd.get("metrics/precision(B)",  0))
            m["box_recall"]     = float(rd.get("metrics/recall(B)",      0))
            m["box_mAP50"]      = float(rd.get("metrics/mAP50(B)",       0))
            m["box_mAP50_95"]   = float(rd.get("metrics/mAP50-95(B)",    0))
            m["pose_mAP50"]     = float(rd.get("metrics/mAP50(P)",       0))
            m["pose_mAP50_95"]  = float(rd.get("metrics/mAP50-95(P)",    0))
            m["val_box_loss"]   = float(rd.get("val/box_loss",           0))
            m["val_pose_loss"]  = float(rd.get("val/pose_loss",          0))

        run_dir = KAGGLE_WORKING / "runs" / "fase1" / run_name
        summary = {
            "experiment_id": exp_id,
            "label":         label,
            "run_name":      run_name,
            "timestamp":     ts,
            "model_weights": exp["weights"],
            "epochs":        exp["epochs"],
            "batch":         exp["batch"],
            "imgsz":         exp["imgsz"],
            "device":        device,
            "status":        "COMPLETED",
            "metrics":       m,
            "run_dir":       str(run_dir),
            "best_weights":  str(run_dir / "weights" / "best.pt"),
        }

        with open(run_dir / "metrics_summary.json", "w") as f:
            json.dump(summary, f, indent=2)

        all_results.append(summary)

        print(f"\n[OK] {label} completato!")
        print(f"     Box  mAP@0.5      : {m.get('box_mAP50',     0):.4f}")
        print(f"     Pose mAP@0.5      : {m.get('pose_mAP50',    0):.4f}")
        print(f"     Box  mAP@0.5:0.95 : {m.get('box_mAP50_95',  0):.4f}")
        print(f"     Pose mAP@0.5:0.95 : {m.get('pose_mAP50_95', 0):.4f}")

    except Exception as e:
        print(f"[ERRORE] {exp_id}: {e}")
        all_results.append({
            "id": exp_id, "label": label,
            "status": "FAILED", "reason": str(e)
        })

    torch.cuda.empty_cache()

print("\n" + "="*60)
print("  TRAINING COMPLETATO")
print("="*60)

In [ ]:
import zipfile

zip_path  = KAGGLE_WORKING / "flypose_sar_results.zip"
completed = [r for r in all_results if r.get("status") == "COMPLETED"]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for r in completed:
        run_dir = Path(r["run_dir"])
        exp_id  = r["experiment_id"]

        for w_name in ["best.pt", "last.pt"]:
            w = run_dir / "weights" / w_name
            if w.exists():
                zf.write(w, f"{exp_id}/weights/{w_name}")
                print(f"  [+] {exp_id}/weights/{w_name}")

        for fname in ["results.csv", "metrics_summary.json"]:
            p = run_dir / fname
            if p.exists():
                zf.write(p, f"{exp_id}/{fname}")
                print(f"  [+] {exp_id}/{fname}")

        for png in run_dir.glob("*.png"):
            zf.write(png, f"{exp_id}/{png.name}")
            print(f"  [+] {exp_id}/{png.name}")

size_mb = zip_path.stat().st_size / 1e6
print(f"\n[OK] ZIP: {zip_path}  ({size_mb:.1f} MB)")
print("Scarica da: pannello Output a destra → flypose_sar_results.zip")